In [7]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
# create a session to reuse the underlying connection
session = requests.Session()

#
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
})

print("setup complete")

setup complete


In [18]:
def scrape_recipe(url):
    try:
        
        response = session.get(url, timeout=10)
        
        # This will raise an error if the site returns a 404 or 500 code
        response.raise_for_status() 
        
        soup = BeautifulSoup(response.text, 'html.parser')
        #print(soup)
        #https://usa.lkk.com/en/recipes/{recipe_path}
        out = {"name": url.split('/')[-1]}
        
        
        boxes = soup.find_all('div', class_='side-box')   

        
        
        for box in boxes:
            heading = box.find('h3')
            
            if(heading):
                #print(heading.text)

                list_items = box.find_all("li")
            
                
                list_items = [x.text for x in list_items]
                #print(list_items)
                out[heading.text] = list_items
                
                
        #print(out)
        # success
        print(f"Successfully scraped: {url}")
        return out
    except requests.exceptions.RequestException as e:
        print(f"Failed to scrape {url}: {e}")

In [16]:
def scrape_recipe_titles(page_num):

    data = []
    
    url = f'https://usa.lkk.com/en/recipes?page={page_num}'
    
    print(f"Fetching main page: {url}")
    response = session.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    recipes = soup.find_all('div', class_='recipe-item')

    
    
    for recipe in recipes:
        
        link_element = recipe.find('a')
        
        if link_element and 'href' in link_element.attrs:

            
            recipe_path = link_element['href']
            
            # Construct the full URL
            full_url = f"https://usa.lkk.com/{recipe_path}"
            
            print(full_url)
            
            data.append(scrape_recipe(full_url))
            
            
            time.sleep(1) 
    return pd.DataFrame(data)

In [20]:

pages = 151
df = scrape_recipe_titles(pages)
print(df.head())

df.to_csv('lkk_recipes.csv', index=False, encoding='utf-8')
print("data saved to lkk_recipes.csv!")

Fetching main page: https://usa.lkk.com/en/recipes?page=151
https://usa.lkk.com//en/recipes/air-fryer-xo-fish-balls-in-honey
Successfully scraped: https://usa.lkk.com//en/recipes/air-fryer-xo-fish-balls-in-honey
https://usa.lkk.com//en/recipes/beef-and-broccoli-stir-fry
Successfully scraped: https://usa.lkk.com//en/recipes/beef-and-broccoli-stir-fry
https://usa.lkk.com//en/recipes/beef-and-broccoli-stir-fry
Successfully scraped: https://usa.lkk.com//en/recipes/beef-and-broccoli-stir-fry
https://usa.lkk.com//en/recipes/pasta-linguine-beef-and-oyster-flavored-sauce
Successfully scraped: https://usa.lkk.com//en/recipes/pasta-linguine-beef-and-oyster-flavored-sauce
https://usa.lkk.com//en/recipes/sauteed-fish-with-black-bean-garlic-sauce
Successfully scraped: https://usa.lkk.com//en/recipes/sauteed-fish-with-black-bean-garlic-sauce
https://usa.lkk.com//en/recipes/seafood-and-tofu-soup
Successfully scraped: https://usa.lkk.com//en/recipes/seafood-and-tofu-soup
https://usa.lkk.com//en/recipe